# Phase 1: Business Understanding

**CRISP-DM Purpose:** Define the business problem, objectives, and success criteria before any data work begins.

**Project:** Safehouse Outcome Drivers — Panel Regression  
**Pipeline:** `pipeline/safehouse_outcome_drivers/`  
**Date:** 2026-04  
**Author:** Pag-asa Analytics

---
## 1.1 Problem Statement

Pag-asa operates 9 safehouses across three Philippine regions. Leadership currently has no evidence-based method to answer the core resource allocation question:

> **Which operational inputs drive better resident outcomes — and by how much?**

This pipeline builds an explanatory panel regression model to answer two specific research questions:

| Model | Research Question | Dependent Variable |
|-------|------------------|-------------------|
| **Model A** | What operational and case-complexity factors explain variance in resident health scores across safehouses and months? | `avg_health_score` (scale 0–10, observed 2.42–3.94) |
| **Model B** | What factors explain variance in resident education progress — and is the observed Visayas gap explained by case complexity or something structural? | `avg_education_progress` (scale 0–100%, observed 13–100) |

### Problem Type
**Explanatory panel regression** — not predictive modeling. The goal is to estimate *partial effects* of each input controlling for all others, not to forecast future outcomes. There is no train/test split; we want the best population-level coefficient estimates from all available data.

### Why Panel Regression?
We have **repeated observations of the same 9 safehouses over ~50 months** (a balanced panel). Ordinary cross-sectional regression would conflate between-safehouse differences (e.g., safehouse 3 always has better leadership) with the true effect of inputs like counseling intensity. A **fixed-effects model** controls for all time-invariant unobserved safehouse characteristics by including a dummy for each safehouse, isolating the within-safehouse variation in inputs and outcomes.

---
## 1.2. Feasibility Assessment

| Criterion | Assessment | Evidence |
|-----------|-----------|----------|
| **Practical Impact** | High — leadership can redirect staff time and budget based on coefficient estimates | Resource allocation is a top-3 operational decision |
| **Data Availability** | Strong — 9 safehouses × ~50 months = ~450 safehouse-month observations; all inputs are pre-recorded in the monthly metrics table | `safehouse_monthly_metrics.csv` contains all primary inputs |
| **Analytical Feasibility** | High — standard fixed-effects OLS is well-suited to this panel structure; statsmodels handles it natively | N=450 is sufficient for 7 predictors + 8 FE dummies |
| **Causal Identification** | Moderate — fixed effects remove safehouse-level confounding, but reverse causality in session/visit counts is a known limitation (addressed explicitly in Phase 5) | See Section 5: Caveats |

### Sample Size Check
Rule of thumb for OLS: 10–20 observations per parameter.  
- Parameters: 7 regression features + 8 safehouse FE dummies + 2 region dummies + intercept = **~18 parameters**  
- Observations: ~450 safehouse-months  
- Ratio: **~25 obs/parameter** ✅ well above minimum

---
## 1.3. Success Criteria

| Criterion | Target | Rationale |
|-----------|--------|----------|
| **R² (Model A — Health)** | ≥ 0.50 | Explains at least half the variance in monthly health scores |
| **R² (Model B — Education)** | ≥ 0.50 | Education has a strong time trend; with `months_since_start` + FEs, R² > 0.50 is achievable |
| **Significant coefficients** | ≥ 2 per model at p < 0.05 with interpretable sign | Minimum for actionable resource allocation guidance |
| **Coefficient signs** | `months_since_start` > 0 for education; `pct_high_risk` < 0 for both | Sanity checks against prior descriptive analysis |
| **Robustness** | HC3 heteroscedasticity-robust standard errors | Panel data typically violates homoscedasticity; HC3 is the conservative choice |

### Baseline to Beat
- **Health**: A null model predicting the grand mean for every row has R² = 0 by construction. The safehouse FE-only model (no operational inputs) is the more relevant baseline — if operational inputs add R² beyond the FEs alone, they are contributing genuine explanatory power.
- **Education**: Same logic, but the time trend (`months_since_start`) alone explains a large share of variance — the question is whether operational inputs explain variance *beyond* the trend.

---
## 1.4. Stakeholder Impact

| Stakeholder | Role | How They Use This Output |
|-------------|------|------------------------|
| **Program managers / regional leads** | Primary operational users | Act on per-safehouse expected vs. actual flags; review underperforming locations |
| **Executive leadership / founders** | Strategic audience | Use coefficient table for resource allocation decisions ("invest more in X, not Y") |
| **Safehouse staff** | Indirect beneficiaries | Better resource allocation improves their capacity to serve residents |

### Decisions Enabled
- **Staffing**: Should we increase counseling session frequency, or is that correlation spurious?
- **Capacity**: Is there an optimal occupancy range we should target?
- **Regional equity**: Is Visayas underperformance explained by case complexity or a structural gap that needs intervention?
- **Performance reviews**: Which safehouses are underperforming *relative to their expected outcomes given their caseload*?

### Human-in-the-Loop Points
- The model flags underperformers; **a program manager reviews** before any intervention is decided
- Coefficient interpretation requires context: the reverse causality caveat (Section 5) must be understood before acting on session/visit coefficients

---
## 1.5. Known Caveats — Reverse Causality

**This section documents the most important analytical limitation upfront, before any modeling occurs.**

### What the Descriptive Data Shows
Prior exploration of the raw data reveals:
- `process_recording_count` has near-zero correlation with outcomes (−0.07 with health, +0.04 with education)
- `home_visitation_count` is *positively* correlated with incidents (+0.20)

### The Reverse Causality Problem
The naive interpretation would be: "more counseling sessions don't improve outcomes" or "home visits *cause* more incidents." Both are almost certainly **backwards**.

The more plausible causal story is:
> Staff *respond to struggling residents* with more sessions and more visits. The causal arrow runs from **poor outcomes → more interventions**, not **more interventions → poor outcomes**.

This is a classic **reverse causality** (simultaneous causality bias) problem inherent in observational panel data. The fixed-effects model controls for time-invariant confounding (e.g., "safehouse 5 always has harder cases") but cannot fully resolve within-safehouse reverse causality.

### Implication for Inference
- We **cannot claim** that reducing counseling sessions would improve health scores.
- We **can claim** that safehouses with systematically higher session intensity (controlling for case complexity and FEs) do not show better outcomes on average — which itself is informative for program design.
- An instrumental variables (IV) approach or a randomized intervention would be needed for causal identification. This is out of scope for this analysis but should be named as a future research direction.

**This caveat will appear explicitly in the coefficient table in Phase 4 and in the executive summary.**

---
## 1.6. Feature Inventory — Hypotheses

Before seeing the regression results, we document our prior hypotheses. This prevents post-hoc rationalization.

| Feature | Expected Sign (Health) | Expected Sign (Education) | Hypothesis |
|---------|----------------------|--------------------------|------------|
| `sessions_per_resident` | Ambiguous (reverse causality) | Ambiguous | May be negative due to reactive staffing |
| `visits_per_resident` | Ambiguous (reverse causality) | Ambiguous | Same |
| `occupancy_rate` | Non-linear (inverted-U) | Negative | Overcrowding hurts outcomes; some threshold effect |
| `pct_high_risk` | Negative | Negative | Higher-risk caseload → harder to achieve high scores |
| `pct_trafficked` | Negative | Negative | Trafficking trauma → more complex recovery |
| `pct_special_needs` | Negative | Negative | Special needs → additional resource demands |
| `months_since_start` | Slight positive | Strong positive | Org learning curve; education shows strong upward trend |
| Region: Visayas (vs. Luzon) | Negative | Negative | Descriptive analysis shows Visayas lags on education |
| Region: Mindanao (vs. Luzon) | Negative | Neutral | Mindanao performs comparably to Luzon on education |

**Phase 4 will test whether these hypotheses are borne out by the data.**

---
## 1.7. Phase Evidence & Assumptions

This section documents the explicit user-confirmed decisions made before Phase 1. Per the anti-hallucination rule, no key design choice proceeds without confirmation.

| Decision | Value | Source |
|----------|-------|--------|
| Problem framing | Explanatory panel regression (no train/test split) | User confirmed Phase 0 |
| Minimum acceptable R² | 0.50 for each model | User confirmed Phase 0 |
| Primary stakeholders | Program managers / regional leads | User confirmed Phase 0 |
| Known data quality issues | None flagged — proceed with standard checks | User confirmed Phase 0 |
| Operationalization sink | CSV + Supabase | User confirmed Phase 0 |
| Model method | OLS + safehouse fixed effects + HC3 robust SEs | User confirmed from initial context |
| Reference region | Luzon (dropped dummy) | User-provided context |
| Reverse causality stance | Acknowledge explicitly; do not overclaim causal direction | User-provided context |

---
## Phase 1 — Sign-off Checklist

Before proceeding to Phase 2 (Data Understanding), confirm:
- [ ] Problem statement accurately describes the org's decision need
- [ ] Two-model structure (health + education) is correct
- [ ] Success criteria (R² ≥ 0.50, ≥ 2 significant coefficients) are appropriate
- [ ] Reverse causality caveat is correctly framed
- [ ] Feature hypothesis table is complete and reasonable

**Awaiting user sign-off to proceed to Phase 2.**

In [1]:
# Phase 1 is documentation-only — no data loading required.
# The following confirms the config paths resolve correctly.

import sys
from pathlib import Path

# Add src/ to path so notebooks can import from src/
sys.path.insert(0, str(Path.cwd().parent))

from src.config import (
    DATASETS_DIR, METRICS_FILE, SAFEHOUSE_FILE, RESIDENTS_FILE,
    OUTCOME_HEALTH, OUTCOME_EDUCATION,
    R2_THRESHOLD_MIN, SIG_COEFS_MIN,
)

print("=== Config paths ===")
print(f"Datasets dir : {DATASETS_DIR}")
print(f"Metrics file : {DATASETS_DIR / METRICS_FILE}  exists={( DATASETS_DIR / METRICS_FILE).exists()}")
print(f"Safehouse file: {DATASETS_DIR / SAFEHOUSE_FILE} exists={(DATASETS_DIR / SAFEHOUSE_FILE).exists()}")
print(f"Residents file: {DATASETS_DIR / RESIDENTS_FILE} exists={(DATASETS_DIR / RESIDENTS_FILE).exists()}")
print()
print("=== Success criteria ===")
print(f"R² minimum   : {R2_THRESHOLD_MIN}")
print(f"Sig. coefs   : ≥ {SIG_COEFS_MIN} at p < 0.05")
print()
print("✅ Phase 1 config check passed — ready for Phase 2.")

=== Config paths ===
Datasets dir : C:\Users\apier\OneDrive\Documents\Code\Intex-II\intex-w26\pipeline\datasets
Metrics file : C:\Users\apier\OneDrive\Documents\Code\Intex-II\intex-w26\pipeline\datasets\safehouse_monthly_metrics.csv  exists=True
Safehouse file: C:\Users\apier\OneDrive\Documents\Code\Intex-II\intex-w26\pipeline\datasets\safehouses.csv exists=True
Residents file: C:\Users\apier\OneDrive\Documents\Code\Intex-II\intex-w26\pipeline\datasets\residents.csv exists=True

=== Success criteria ===
R² minimum   : 0.5
Sig. coefs   : ≥ 2 at p < 0.05

✅ Phase 1 config check passed — ready for Phase 2.
